# Week 6 - SEO Analysis with Python

## Course Overview

Welcome! This week, we'll learn how to use Python to analyze **Search Engine Optimization (SEO)** data. By the end of this lesson, you'll be able to:

- **Collect** search engine results (SERP data) using an API
- **Vectorize** page titles using TF-IDF to convert text into numbers
- **Discover topics** in search results using Non-negative Matrix Factorization (NMF)
- **Predict** which topics are associated with higher or lower search rankings using linear regression

**Why SEO?** Search Engine Optimization is one of the most important digital marketing strategies. Companies invest heavily in understanding what content ranks well on Google. By analyzing SERP data programmatically, we can uncover patterns that manual analysis might miss.

**The Goal:** We want to answer: *What topics in page titles are associated with higher Google rankings?* To answer this, we'll collect search results for AI marketing-related keywords and apply NLP and regression techniques.

## 1. Setup and Installation

First, let's install and import all the packages we'll need for this analysis.

| Package | Purpose |
|---------|--------|
| `requests` | Making API calls to retrieve search results |
| `pandas` | Data manipulation and analysis |
| `sklearn` | TF-IDF vectorization and NMF topic modeling |
| `pingouin` | Statistical analysis (linear regression) |

In [ ]:
# Install required packages
!pip3 install pingouin

In [ ]:
# Import all required libraries
import requests
import time
from urllib.parse import urlparse

import pandas as pd
import pingouin as pg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

## 2. Collecting SERP Data

### 2.1 What is SERP?

**SERP** stands for **Search Engine Results Page** — it's the page you see after typing a query into Google. Each result on the page has:

- **Title**: The clickable headline
- **Snippet**: The short description below the title
- **URL**: The link to the webpage
- **Rank (Position)**: Where the result appears on the page (1 = top result)

**Why does rank matter?** Studies show that the first result on Google gets about 30% of all clicks, the second gets about 15%, and results on page 2 get almost no clicks. Understanding what drives higher rankings is critical for any digital marketing strategy.

### 2.2 Using SerpAPI to Collect Search Results

We'll use [SerpAPI](https://serpapi.com/) to programmatically retrieve Google search results. This API allows us to:
- Search for multiple keywords
- Collect up to 200 results per keyword
- Get structured data (title, snippet, URL, rank) for each result

We search for three AI marketing-related keywords:
1. **"AI marketing"** — broad industry term
2. **"AI Promotion"** — focused on promotional tools
3. **"AI Copy Generator"** — focused on content creation tools

> **Note:** You will need a SerpAPI key to run this cell. You can get a free trial key at [serpapi.com](https://serpapi.com/). Store your API key securely using Google Colab's `userdata` feature (Secrets panel on the left sidebar) rather than hardcoding it in your notebook.

In [ ]:
# --- API Key Setup ---
# Option 1: Google Colab (recommended) - store your key in Colab Secrets
# from google.colab import userdata
# API_KEY = userdata.get('SERPAPI_KEY')

# Option 2: Replace with your own API key for local use
API_KEY = "YOUR_API_KEY_HERE"

# --- Search Configuration ---
KEYWORDS = ["AI marketing", "AI Promotion", "AI Copy Generator"]
RESULTS_PER_PAGE = 100  # Max results per page allowed by SerpAPI
MAX_RESULTS = 200       # Total results we want per keyword

all_results = []

for keyword in KEYWORDS:
    total_collected = 0
    start = 0

    while total_collected < MAX_RESULTS:
        # Build the API request parameters
        params = {
            "engine": "google",
            "q": keyword,
            "api_key": API_KEY,
            "hl": "en",      # Language: English
            "gl": "us",      # Country: United States
            "num": RESULTS_PER_PAGE,
            "start": start   # Pagination offset
        }

        # Make the API call
        response = requests.get("https://serpapi.com/search", params=params)
        data = response.json()

        # Check if we got results
        if "organic_results" not in data:
            print(f"No more results for keyword: {keyword}")
            break

        # Extract metadata
        search_info = data.get("search_information", {})
        search_metadata = data.get("search_metadata", {})
        total_results = search_info.get("total_results")
        latency = search_metadata.get("total_time_taken")

        # Extract each result
        for result in data["organic_results"]:
            link = result.get("link")
            all_results.append({
                "keyword": keyword,
                "rank": result.get("position"),
                "title": result.get("title"),
                "snippet": result.get("snippet"),
                "url": link,
                "domain": urlparse(link).netloc if link else None,
                "total_results": total_results,
                "latency": latency,
                "location": "us",
                "language": "en"
            })

        total_collected += len(data["organic_results"])
        start += RESULTS_PER_PAGE
        time.sleep(2)  # Avoid hitting rate limits

# Create DataFrame
df = pd.DataFrame(all_results)

# Add domain popularity (how often a domain appears across all results)
df['domain_popularity'] = df['domain'].map(df['domain'].value_counts())

print(f"Collected {len(df)} search results across {len(KEYWORDS)} keywords.")
df.head()

### 2.3 Saving the Data

It's good practice to save your collected data to a CSV file. This way, you don't need to re-run the API calls every time — which saves both time and API credits.

In [ ]:
# Save the collected data to CSV
df.to_csv("serp.csv", index=False)
print("Data saved to serp.csv")

> **If you don't have an API key**, you can load the pre-collected data instead:
> ```python
> df = pd.read_csv("serp.csv")
> ```

In [ ]:
 df = pd.read_csv("serp.csv")

## 3. Text Vectorization with TF-IDF

### 3.1 Why Vectorize Page Titles?

We want to understand what **topics** appear in search result titles and whether certain topics are associated with higher rankings. But computers can't analyze raw text directly — we need to convert it into numbers first.

As we learned in Week 3, **TF-IDF (Term Frequency–Inverse Document Frequency)** is a powerful method for converting text into numerical features:

- **TF (Term Frequency):** How often a word appears in a document
- **IDF (Inverse Document Frequency):** How rare a word is across all documents
- **TF-IDF = TF × IDF:** Words that are frequent in one document but rare across all documents get the highest scores

**In our context:** Each search result **title** is a document. Words that are distinctive to certain titles (rather than appearing everywhere) will have higher TF-IDF scores.

### 3.2 Applying TF-IDF to SERP Titles

We configure the vectorizer with:
- `max_df=0.95` — Ignore words appearing in more than 95% of titles (too common to be informative)
- `min_df=2` — Ignore words appearing in fewer than 2 titles (too rare to generalize)
- `stop_words='english'` — Remove common English words ("the", "is", "and", etc.)

In [ ]:
# Extract the titles as our documents
docs = df['title']

# Create and fit the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
tfidf = tfidf_vectorizer.fit_transform(docs)

# Convert to a readable DataFrame
tfidf_df = pd.DataFrame(tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

print(f"TF-IDF matrix shape: {tfidf_df.shape[0]} documents x {tfidf_df.shape[1]} unique terms")
tfidf_df

**Interpreting the TF-IDF Matrix:**

- Each **row** represents a search result title
- Each **column** represents a unique word
- Each **value** is the TF-IDF score — higher means the word is more important to that title
- Most values are **0.0** because any given title only contains a small subset of all words (this is a sparse matrix)

## 4. Topic Modeling with NMF

### 4.1 Why Topic Modeling?

Our TF-IDF matrix has hundreds of columns (one per word). That's too many dimensions to interpret directly. **Topic modeling** reduces this complexity by grouping related words into a smaller number of meaningful "topics."

As we learned in Week 3, **Non-negative Matrix Factorization (NMF)** decomposes our document-term matrix into:

$$M = W \times H$$

Where:
- **M**: Original TF-IDF matrix (documents × terms)
- **W**: Document-topic matrix — *How strongly does each title belong to each topic?*
- **H**: Topic-term matrix — *Which words define each topic?*

### 4.2 Applying NMF

We choose **7 topics** to capture the diversity of AI marketing-related search results. The top 4 words for each topic will serve as the topic label.

In [ ]:
# Step 1: Create and fit the NMF model with 7 topics
nmf_model = NMF(n_components=7, random_state=0)
W = nmf_model.fit_transform(tfidf)  # W = document-topic matrix

# Step 2: Extract topic labels from top words
feature_names = tfidf_vectorizer.get_feature_names_out()
topic_names = []

print("Discovered Topics:\n")
for topic_index, topic in enumerate(nmf_model.components_):
    # Get the top 4 words for this topic using pandas Series
    top_words = pd.Series(topic, index=feature_names).nlargest(4).index.tolist()
    top_words_string = " ".join(top_words)
    print("Topic #" + str(topic_index) + ": " + top_words_string)
    topic_names.append(top_words_string)

# Step 3: Create a document-topic DataFrame
topic_df = pd.DataFrame(W, columns=topic_names)

print("\nDocument-Topic Matrix (W):")
topic_df

**Interpreting the Topics:**

Each topic is defined by its top words. For example:
- A topic with words like "marketing", "ai", "digital", "future" represents content about **AI in digital marketing strategy**
- A topic with words like "tools", "best", "2025", "copywriting" represents **tool comparison/review content**
- A topic with words like "promo", "ai", "codes", "video" represents **promotional and video content**

Each row in the W matrix shows how strongly a search result belongs to each topic. Higher values mean the title is more closely related to that topic.

## 5. Linear Regression: Which Topics Rank Higher?

### 5.1 The Business Question

Now we have topics extracted from search result titles and the **rank** (position) of each result on Google. The key question is:

> **Which topics are associated with higher Google rankings?**

If we find that certain topics tend to appear in higher-ranked results, this gives us actionable SEO insights — we should create content that emphasizes those topics!

### 5.2 Understanding the Model

We use **linear regression** to model the relationship:

$$\text{Rank} = \beta_0 + \beta_1 \cdot \text{Topic}_1 + \beta_2 \cdot \text{Topic}_2 + \dots + \beta_7 \cdot \text{Topic}_7$$

**Important note about interpreting rank:** A **lower rank number is better** (rank 1 = top of the page). So:

| Coefficient | Interpretation |
|-------------|---------------|
| **Negative** ($\beta < 0$) | This topic is associated with **better rankings** (lower rank number) |
| **Positive** ($\beta > 0$) | This topic is associated with **worse rankings** (higher rank number) |
| **Near zero** | This topic has little effect on ranking |

We also check the **p-value** to determine statistical significance:
- **p < 0.05**: The relationship is statistically significant
- **p >= 0.05**: The relationship is not statistically significant (could be due to chance)

### 5.3 Basic Model: Topics Only

Let's start with a simple model that uses only the topic weights to predict rank.

In [ ]:
# Step 1: Combine topic weights (X) with rank (Y)
df_model = topic_df.copy()
df_model['rank'] = df['rank'].values

# Step 2: Run linear regression (topics -> rank)
result = pg.linear_regression(
    df_model.drop(columns='rank'),  # X: topic weights
    df_model['rank']                # Y: search rank
)

# Step 3: Display key results
print("Basic Model: Topics -> Rank\n")
print(result[['names', 'coef', 'pval']].round(3))

print("\nHow to read this:")
print("- Negative 'coef' = this topic is associated with BETTER rankings (lower rank number)")
print("- Positive 'coef' = this topic is associated with WORSE rankings")
print("- 'pval' < 0.05 = the effect is statistically significant")

### 5.4 Full Model: Topics + Keyword + Domain Popularity

The basic model only considers topics. But other factors might also influence ranking:

- **Keyword searched:** Different keywords may have different ranking distributions
- **Domain popularity:** Domains that appear frequently across results may have an inherent ranking advantage (e.g., YouTube, Wikipedia)

Let's build a more comprehensive model that includes these additional variables.

> **Note:** We use **dummy variables** (also called one-hot encoding) to convert the categorical `keyword` column into numeric columns. We drop one keyword category to avoid the [dummy variable trap](https://en.wikipedia.org/wiki/Dummy_variable_(statistics)) (perfect multicollinearity).

In [ ]:
# Step 1: Combine original data with topic weights
combined_df = pd.concat([df, topic_df], axis=1)

# Step 2: Create dummy variables for the keyword column
keyword_dummies = pd.get_dummies(combined_df['keyword'], prefix='keyword').astype(int)
combined_df = pd.concat([combined_df, keyword_dummies], axis=1)

# Step 3: Select features for the full model
# We include: keyword dummies (drop one to avoid multicollinearity), domain popularity, and all topics
x = combined_df[['keyword_AI Promotion', 'keyword_AI marketing', 'domain_popularity'] + topic_names]
y = combined_df['rank']

# Step 4: Run the full regression
result_full = pg.linear_regression(x, y)

# Step 5: Display key results
print("Full Model: Topics + Keyword + Domain Popularity -> Rank\n")
print(result_full[['names', 'coef', 'pval']].round(3))

print("\nHow to read this:")
print("- Negative 'coef' = associated with BETTER rankings (lower rank number)")
print("- Positive 'coef' = associated with WORSE rankings")
print("- 'pval' < 0.05 = statistically significant")